# Hyperparameter Tuning for Citation Impact Prediction

This notebook optimizes hyperparameters for the best-performing models:
- **LightGBM Classifier** (HighImpact prediction)
- **LightGBM Regressor** (Citations_log prediction)

**Strategy:**
- Use RandomizedSearchCV (more efficient than GridSearch)
- 5-fold cross-validation on random split (temporal split too imbalanced)
- Focus on key hyperparameters: n_estimators, max_depth, learning_rate, num_leaves
- Compare tuned vs baseline performance

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import json
from datetime import datetime
import time

# LightGBM
import lightgbm as lgb

# Hyperparameter tuning
from sklearn.model_selection import RandomizedSearchCV, cross_val_score
from scipy.stats import randint, uniform

# Evaluation metrics
from sklearn.metrics import (
    make_scorer, f1_score, roc_auc_score,
    mean_squared_error, r2_score
)

# Set random seed
np.random.seed(42)

# Plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries loaded successfully!")

## 1. Load Data and Baseline Results

In [ ]:
# Load random split data (use for tuning)
print("Loading training data...")
X_train = pd.read_csv('../data/X_train_random.csv')
X_test = pd.read_csv('../data/X_test_random.csv')
y_reg_train = pd.read_csv('../data/y_reg_train_random.csv')['Citations_log']
y_reg_test = pd.read_csv('../data/y_reg_test_random.csv')['Citations_log']
y_cls_train = pd.read_csv('../data/y_cls_train_random.csv')['HighImpact']
y_cls_test = pd.read_csv('../data/y_cls_test_random.csv')['HighImpact']

print(f"✓ Data loaded")
print(f"  Train: {X_train.shape}, Test: {X_test.shape}")
print(f"  Features: {X_train.shape[1]}")
print(f"  Class balance: {y_cls_train.value_counts(normalize=True).to_dict()}")

In [ ]:
# Load baseline results for comparison
baseline_cls = pd.read_csv('../results/classification_results.csv')
baseline_reg = pd.read_csv('../results/regression_results.csv')

# Get LightGBM baseline scores (random split)
lgb_cls_baseline = baseline_cls[baseline_cls['model'] == 'LightGBM (Random Split)'].iloc[0]
lgb_reg_baseline = baseline_reg[baseline_reg['model'] == 'LightGBM (Random Split)'].iloc[0]

print("Baseline Performance (LightGBM):")
print(f"\nClassification:")
print(f"  F1-Score: {lgb_cls_baseline['f1_score']:.4f}")
print(f"  AUC-ROC:  {lgb_cls_baseline['auc_roc']:.4f}")

print(f"\nRegression:")
print(f"  RMSE: {lgb_reg_baseline['rmse']:.4f}")
print(f"  R²:   {lgb_reg_baseline['r2']:.4f}")

## 2. Hyperparameter Tuning - Classification

**Goal**: Optimize LightGBM Classifier for F1-score

**Key Hyperparameters:**
- `n_estimators`: Number of boosting rounds (50-300)
- `max_depth`: Maximum tree depth (5-20)
- `learning_rate`: Step size (0.01-0.3)
- `num_leaves`: Maximum leaves per tree (20-150)
- `min_child_samples`: Minimum samples in leaf (5-50)
- `subsample`: Row sampling ratio (0.6-1.0)
- `colsample_bytree`: Column sampling ratio (0.6-1.0)

In [ ]:
# Calculate scale_pos_weight for class imbalance
scale_pos_weight = (y_cls_train == 0).sum() / (y_cls_train == 1).sum()
print(f"Class imbalance ratio (scale_pos_weight): {scale_pos_weight:.2f}")

In [ ]:
# Define parameter distribution for RandomizedSearchCV
param_dist_cls = {
    'n_estimators': randint(50, 300),
    'max_depth': randint(5, 20),
    'learning_rate': uniform(0.01, 0.29),  # 0.01 to 0.30
    'num_leaves': randint(20, 150),
    'min_child_samples': randint(5, 50),
    'subsample': uniform(0.6, 0.4),  # 0.6 to 1.0
    'colsample_bytree': uniform(0.6, 0.4),  # 0.6 to 1.0
    'reg_alpha': uniform(0, 1),  # L1 regularization
    'reg_lambda': uniform(0, 1),  # L2 regularization
}

print("Parameter search space defined:")
for param, dist in param_dist_cls.items():
    print(f"  {param}: {dist}")

In [ ]:
print("="*80)
print("HYPERPARAMETER TUNING: LightGBM Classifier")
print("="*80)

# Create base model
lgb_cls_base = lgb.LGBMClassifier(
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

# Create F1-score scorer
f1_scorer = make_scorer(f1_score)

# RandomizedSearchCV
random_search_cls = RandomizedSearchCV(
    estimator=lgb_cls_base,
    param_distributions=param_dist_cls,
    n_iter=50,  # Try 50 random combinations
    scoring=f1_scorer,
    cv=5,  # 5-fold cross-validation
    verbose=2,
    random_state=42,
    n_jobs=-1
)

# Run search
print("\nStarting hyperparameter search (50 iterations x 5 folds = 250 fits)...")
print("This may take 10-20 minutes...\n")

start_time = time.time()
random_search_cls.fit(X_train, y_cls_train)
elapsed_time = time.time() - start_time

print(f"\n✓ Tuning complete in {elapsed_time/60:.1f} minutes")

In [ ]:
# Best parameters
print("\nBest Hyperparameters (Classification):")
for param, value in random_search_cls.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest CV F1-Score: {random_search_cls.best_score_:.4f}")
print(f"Baseline F1-Score: {lgb_cls_baseline['f1_score']:.4f}")
improvement = (random_search_cls.best_score_ - lgb_cls_baseline['f1_score']) / lgb_cls_baseline['f1_score'] * 100
print(f"Improvement: {improvement:+.2f}%")

In [ ]:
# Evaluate best model on test set
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

best_lgb_cls = random_search_cls.best_estimator_

y_pred_cls = best_lgb_cls.predict(X_test)
y_proba_cls = best_lgb_cls.predict_proba(X_test)[:, 1]

print("\nTest Set Performance (Tuned Model):")
print(f"  Accuracy:  {accuracy_score(y_cls_test, y_pred_cls):.4f}")
print(f"  Precision: {precision_score(y_cls_test, y_pred_cls):.4f}")
print(f"  Recall:    {recall_score(y_cls_test, y_pred_cls):.4f}")
print(f"  F1-Score:  {f1_score(y_cls_test, y_pred_cls):.4f}")
print(f"  AUC-ROC:   {roc_auc_score(y_cls_test, y_proba_cls):.4f}")

print("\nBaseline Test Set Performance:")
print(f"  Accuracy:  {lgb_cls_baseline['accuracy']:.4f}")
print(f"  Precision: {lgb_cls_baseline['precision']:.4f}")
print(f"  Recall:    {lgb_cls_baseline['recall']:.4f}")
print(f"  F1-Score:  {lgb_cls_baseline['f1_score']:.4f}")
print(f"  AUC-ROC:   {lgb_cls_baseline['auc_roc']:.4f}")

## 3. Hyperparameter Tuning - Regression

**Goal**: Optimize LightGBM Regressor for R² score

In [ ]:
# Define parameter distribution for regression
param_dist_reg = {
    'n_estimators': randint(50, 300),
    'max_depth': randint(5, 20),
    'learning_rate': uniform(0.01, 0.29),
    'num_leaves': randint(20, 150),
    'min_child_samples': randint(5, 50),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'reg_alpha': uniform(0, 1),
    'reg_lambda': uniform(0, 1),
}

print("Parameter search space defined for regression")

In [ ]:
print("="*80)
print("HYPERPARAMETER TUNING: LightGBM Regressor")
print("="*80)

# Create base model
lgb_reg_base = lgb.LGBMRegressor(
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

# Create R² scorer
r2_scorer = make_scorer(r2_score)

# RandomizedSearchCV
random_search_reg = RandomizedSearchCV(
    estimator=lgb_reg_base,
    param_distributions=param_dist_reg,
    n_iter=50,
    scoring=r2_scorer,
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

# Run search
print("\nStarting hyperparameter search (50 iterations x 5 folds = 250 fits)...")
print("This may take 10-20 minutes...\n")

start_time = time.time()
random_search_reg.fit(X_train, y_reg_train)
elapsed_time = time.time() - start_time

print(f"\n✓ Tuning complete in {elapsed_time/60:.1f} minutes")

In [ ]:
# Best parameters
print("\nBest Hyperparameters (Regression):")
for param, value in random_search_reg.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest CV R²: {random_search_reg.best_score_:.4f}")
print(f"Baseline R²: {lgb_reg_baseline['r2']:.4f}")
improvement = (random_search_reg.best_score_ - lgb_reg_baseline['r2']) / lgb_reg_baseline['r2'] * 100
print(f"Improvement: {improvement:+.2f}%")

In [ ]:
# Evaluate best model on test set
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

best_lgb_reg = random_search_reg.best_estimator_

y_pred_reg = best_lgb_reg.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_reg_test, y_pred_reg))
mae = mean_absolute_error(y_reg_test, y_pred_reg)
r2 = r2_score(y_reg_test, y_pred_reg)

print("\nTest Set Performance (Tuned Model):")
print(f"  RMSE: {rmse:.4f}")
print(f"  MAE:  {mae:.4f}")
print(f"  R²:   {r2:.4f}")

print("\nBaseline Test Set Performance:")
print(f"  RMSE: {lgb_reg_baseline['rmse']:.4f}")
print(f"  MAE:  {lgb_reg_baseline['mae']:.4f}")
print(f"  R²:   {lgb_reg_baseline['r2']:.4f}")

## 4. Comparison: Baseline vs Tuned

In [ ]:
# Create comparison DataFrame
comparison_cls = pd.DataFrame([
    {
        'Model': 'Baseline',
        'Accuracy': lgb_cls_baseline['accuracy'],
        'Precision': lgb_cls_baseline['precision'],
        'Recall': lgb_cls_baseline['recall'],
        'F1-Score': lgb_cls_baseline['f1_score'],
        'AUC-ROC': lgb_cls_baseline['auc_roc']
    },
    {
        'Model': 'Tuned',
        'Accuracy': accuracy_score(y_cls_test, y_pred_cls),
        'Precision': precision_score(y_cls_test, y_pred_cls),
        'Recall': recall_score(y_cls_test, y_pred_cls),
        'F1-Score': f1_score(y_cls_test, y_pred_cls),
        'AUC-ROC': roc_auc_score(y_cls_test, y_proba_cls)
    }
])

comparison_reg = pd.DataFrame([
    {
        'Model': 'Baseline',
        'RMSE': lgb_reg_baseline['rmse'],
        'MAE': lgb_reg_baseline['mae'],
        'R²': lgb_reg_baseline['r2']
    },
    {
        'Model': 'Tuned',
        'RMSE': rmse,
        'MAE': mae,
        'R²': r2
    }
])

print("="*80)
print("COMPARISON: Baseline vs Tuned Models")
print("="*80)

print("\nClassification (HighImpact):")
print(comparison_cls.to_string(index=False))

print("\nRegression (Citations_log):")
print(comparison_reg.to_string(index=False))

In [ ]:
# Visualize improvements
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Classification metrics
metrics_cls = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
baseline_vals_cls = comparison_cls[comparison_cls['Model'] == 'Baseline'][metrics_cls].values[0]
tuned_vals_cls = comparison_cls[comparison_cls['Model'] == 'Tuned'][metrics_cls].values[0]

x = np.arange(len(metrics_cls))
width = 0.35

axes[0].bar(x - width/2, baseline_vals_cls, width, label='Baseline', alpha=0.8)
axes[0].bar(x + width/2, tuned_vals_cls, width, label='Tuned', alpha=0.8)
axes[0].set_ylabel('Score')
axes[0].set_title('Classification: Baseline vs Tuned')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics_cls, rotation=45, ha='right')
axes[0].legend()
axes[0].set_ylim([0, 1])
axes[0].grid(axis='y', alpha=0.3)

# Regression metrics (normalize for visualization)
metrics_reg = ['RMSE\n(inverted)', 'MAE\n(inverted)', 'R²']
baseline_vals_reg = [1 - lgb_reg_baseline['rmse'], 1 - lgb_reg_baseline['mae'], lgb_reg_baseline['r2']]
tuned_vals_reg = [1 - rmse, 1 - mae, r2]

x = np.arange(len(metrics_reg))

axes[1].bar(x - width/2, baseline_vals_reg, width, label='Baseline', alpha=0.8)
axes[1].bar(x + width/2, tuned_vals_reg, width, label='Tuned', alpha=0.8)
axes[1].set_ylabel('Score (higher = better)')
axes[1].set_title('Regression: Baseline vs Tuned')
axes[1].set_xticks(x)
axes[1].set_xticklabels(metrics_reg)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/hyperparameter_tuning_comparison.png', dpi=300, bbox_inches='tight')
print("\n✓ Comparison plot saved to results/hyperparameter_tuning_comparison.png")
plt.show()

## 5. Save Tuned Models

In [ ]:
import os

# Create directory for tuned models
os.makedirs('../models/tuned', exist_ok=True)

print("Saving tuned models...")

# Save tuned classifier
with open('../models/tuned/lightgbm_classifier_tuned.pkl', 'wb') as f:
    pickle.dump(best_lgb_cls, f)
print("✓ Saved tuned classifier")

# Save tuned regressor
with open('../models/tuned/lightgbm_regressor_tuned.pkl', 'wb') as f:
    pickle.dump(best_lgb_reg, f)
print("✓ Saved tuned regressor")

# Save hyperparameters
tuning_results = {
    'classification': {
        'best_params': random_search_cls.best_params_,
        'best_cv_score': random_search_cls.best_score_,
        'test_metrics': {
            'accuracy': float(accuracy_score(y_cls_test, y_pred_cls)),
            'precision': float(precision_score(y_cls_test, y_pred_cls)),
            'recall': float(recall_score(y_cls_test, y_pred_cls)),
            'f1_score': float(f1_score(y_cls_test, y_pred_cls)),
            'auc_roc': float(roc_auc_score(y_cls_test, y_proba_cls))
        }
    },
    'regression': {
        'best_params': random_search_reg.best_params_,
        'best_cv_score': random_search_reg.best_score_,
        'test_metrics': {
            'rmse': float(rmse),
            'mae': float(mae),
            'r2': float(r2)
        }
    },
    'tuning_info': {
        'n_iterations': 50,
        'cv_folds': 5,
        'total_fits': 250,
        'date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }
}

with open('../models/tuned/tuning_results.json', 'w') as f:
    json.dump(tuning_results, f, indent=2)
print("✓ Saved tuning results")

# Save comparison DataFrames
comparison_cls.to_csv('../results/tuning_comparison_classification.csv', index=False)
comparison_reg.to_csv('../results/tuning_comparison_regression.csv', index=False)
print("✓ Saved comparison tables")

print("\nAll tuned models and results saved to:")
print("  models/tuned/lightgbm_classifier_tuned.pkl")
print("  models/tuned/lightgbm_regressor_tuned.pkl")
print("  models/tuned/tuning_results.json")
print("  results/tuning_comparison_*.csv")

## 6. Summary and Recommendations

In [ ]:
print("="*80)
print("HYPERPARAMETER TUNING COMPLETE")
print("="*80)

print("\n📊 PERFORMANCE IMPROVEMENTS:")

# Classification improvements
f1_improvement = (f1_score(y_cls_test, y_pred_cls) - lgb_cls_baseline['f1_score']) / lgb_cls_baseline['f1_score'] * 100
auc_improvement = (roc_auc_score(y_cls_test, y_proba_cls) - lgb_cls_baseline['auc_roc']) / lgb_cls_baseline['auc_roc'] * 100

print("\n  Classification (HighImpact):")
print(f"    F1-Score:  {lgb_cls_baseline['f1_score']:.4f} → {f1_score(y_cls_test, y_pred_cls):.4f} ({f1_improvement:+.2f}%)")
print(f"    AUC-ROC:   {lgb_cls_baseline['auc_roc']:.4f} → {roc_auc_score(y_cls_test, y_proba_cls):.4f} ({auc_improvement:+.2f}%)")

# Regression improvements
r2_improvement = (r2 - lgb_reg_baseline['r2']) / lgb_reg_baseline['r2'] * 100
rmse_improvement = (lgb_reg_baseline['rmse'] - rmse) / lgb_reg_baseline['rmse'] * 100  # Lower is better

print("\n  Regression (Citations_log):")
print(f"    R²:    {lgb_reg_baseline['r2']:.4f} → {r2:.4f} ({r2_improvement:+.2f}%)")
print(f"    RMSE:  {lgb_reg_baseline['rmse']:.4f} → {rmse:.4f} ({rmse_improvement:+.2f}% reduction)")

print("\n📂 OUTPUT FILES:")
print("  ✓ models/tuned/lightgbm_classifier_tuned.pkl")
print("  ✓ models/tuned/lightgbm_regressor_tuned.pkl")
print("  ✓ models/tuned/tuning_results.json")
print("  ✓ results/tuning_comparison_classification.csv")
print("  ✓ results/tuning_comparison_regression.csv")
print("  ✓ results/hyperparameter_tuning_comparison.png")

print("\n🎯 RECOMMENDATION:")
if f1_improvement > 2 or r2_improvement > 2:
    print("  ✅ Use TUNED models for deployment (significant improvement)")
else:
    print("  ⚠️  Improvements are marginal (<2%)")
    print("  ✅ Use TUNED models (slight edge) or baseline (simpler)")

print("\n🚀 NEXT STEPS:")
print("  1. Deploy tuned models to Streamlit app")
print("  2. Test predictions with real papers")
print("  3. Create model insights dashboard")
print("  4. Document findings in final report")

print("\n" + "="*80)